In [16]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV, cross_val_score

from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.model_selection import KFold
from scipy.stats import randint

import joblib
from pathlib import Path

print("Imports completed successfully!")

Imports completed successfully!


In [17]:
train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/validation.parquet")
test_df = pd.read_parquet("../data/test.parquet")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (58516, 45)
Validation: (19506, 45)
Testing: (19506, 45)


In [18]:
# Create category and genre counts
for df in [train_df, val_df, test_df]:
    df["categories_count"] = df["categories"].apply(len)
    df["genres_count"] = df["genres"].apply(len)

# Use the same features as M3
features = [
    "price",
    "windows",
    "mac",
    "linux",
    "required_age",
    "categories_count",
    "genres_count"
]

# Prepare training data
X_train = train_df[features]
y_train = np.log1p(train_df["positive"])

# Prepare validation data
X_val = val_df[features]
y_val = np.log1p(val_df["positive"])

# Prepare test data (we will use it later)
X_test = test_df[features]
y_test = np.log1p(test_df["positive"])

print("Features:", features)
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Target:", "log1p(positive)")

Features: ['price', 'windows', 'mac', 'linux', 'required_age', 'categories_count', 'genres_count']
Training shape: (58516, 7)
Validation shape: (19506, 7)
Target: log1p(positive)


In [19]:
# Create a shallow Decision Tree
readable_tree = DecisionTreeRegressor(
    max_depth=3,
    random_state=42
)

# Train using training data only
readable_tree.fit(X_train, y_train)

# Print the tree's questions and leaf predictions
tree_rules = export_text(
    readable_tree,
    feature_names=features
)

print(tree_rules)

|--- categories_count <= 4.50
|   |--- genres_count <= 0.50
|   |   |--- price <= 0.29
|   |   |   |--- value: [0.01]
|   |   |--- price >  0.29
|   |   |   |--- value: [4.23]
|   |--- genres_count >  0.50
|   |   |--- categories_count <= 2.50
|   |   |   |--- value: [1.83]
|   |   |--- categories_count >  2.50
|   |   |   |--- value: [2.56]
|--- categories_count >  4.50
|   |--- required_age <= 8.50
|   |   |--- categories_count <= 10.50
|   |   |   |--- value: [3.66]
|   |   |--- categories_count >  10.50
|   |   |   |--- value: [5.01]
|   |--- required_age >  8.50
|   |   |--- genres_count <= 3.50
|   |   |   |--- value: [8.45]
|   |   |--- genres_count >  3.50
|   |   |   |--- value: [5.02]



The decision tree first splits games based on their number of categories, which suggests that this feature is useful for predicting positive reviews. It then uses features such as the number of genres, price and required age to divide games into smaller groups. The values at the leaves represent predicted values on the log1p(positive) scale, rather than the original number of reviews. Since the tree has a maximum depth of 3, it is relatively easy to understand, but it may not capture all the patterns in the dataset.

In [20]:
# Combine training and validation data
development_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

# Prepare features and target
X_dev = development_df[features]
y_dev = np.log1p(development_df["positive"])

# Use the same five folds for all models
cv = KFold(
    n_splits=5,
    shuffle=False
)

print("Development data:", X_dev.shape)
print("Target:", y_dev.shape)
print("Cross-validation folds:", cv.get_n_splits())

Development data: (78022, 7)
Target: (78022,)
Cross-validation folds: 5


In [21]:
# Decision Tree hyperparameter search

tree_parameters = {
    "max_depth": [3, 5, 8, 12, 20, None],
    "min_samples_split": randint(2, 30),
    "min_samples_leaf": randint(1, 30)
}

tree_search = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=tree_parameters,
    n_iter=15,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

tree_search.fit(X_dev, y_dev)

best_tree = tree_search.best_estimator_

print("Best Decision Tree parameters:")
print(tree_search.best_params_)

print("\nBest Decision Tree CV R²:")
print(round(tree_search.best_score_, 4))

Best Decision Tree parameters:
{'max_depth': 12, 'min_samples_leaf': 26, 'min_samples_split': 23}

Best Decision Tree CV R²:
0.2691


In [22]:
# Random Forest hyperparameter search

forest_parameters = {
    "n_estimators": [100, 150],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 15),
    "max_features": [1.0, "sqrt", 0.7]
}

forest_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ),
    param_distributions=forest_parameters,
    n_iter=8,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

forest_search.fit(X_dev, y_dev)

best_forest = forest_search.best_estimator_

print("Best Random Forest parameters:")
print(forest_search.best_params_)

print("\nBest Random Forest CV R²:")
print(round(forest_search.best_score_, 4))

Best Random Forest parameters:
{'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 10, 'min_samples_split': 12, 'n_estimators': 100}

Best Random Forest CV R²:
0.2866


In [23]:
# Create the original M3 Linear Regression model
linear_model = LinearRegression()

# Store all three models
models = {
    "Linear Regression": linear_model,
    "Decision Tree": best_tree,
    "Random Forest": best_forest
}

# Compare using the same five folds and R²
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_dev,
        y_dev,
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )

    cv_results[name] = {
        "Mean R²": scores.mean(),
        "Std R²": scores.std()
    }

# Display results
comparison_df = pd.DataFrame(cv_results).T

print(comparison_df.round(4))

                   Mean R²  Std R²
Linear Regression   0.2141  0.0066
Decision Tree       0.2691  0.0090
Random Forest       0.2866  0.0090


In [24]:
# Select Random Forest as the winning model
best_model = best_forest

# Train on all development data
best_model.fit(X_dev, y_dev)

# Predict on the test data
y_pred = best_model.predict(X_test)

# Calculate evaluation metrics
test_r2 = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)

print("Winning model: Random Forest")
print("Test R²:", round(test_r2, 4))
print("Test MAE:", round(test_mae, 4))

Winning model: Random Forest
Test R²: 0.2921
Test MAE: 1.5767


The Random Forest was selected as the winning model because it achieved the highest cross-validation R² score among the three models. On the test dataset, it achieved an R² of 0.2921 and an MAE of 1.5767. Compared with the Linear Regression model from Milestone 3, the Random Forest improved the R² from 0.2179 to 0.2921 and reduced the MAE from 1.7118 to 1.5767. This indicates that the Random Forest performs better at predicting the logarithm of positive reviews.

In [25]:
# Save the winning review prediction model

model_artifact = {
    "model": best_model,
    "features": features,
    "model_name": "Random Forest Regressor",
    "best_parameters": forest_search.best_params_,
    "cv_r2": forest_search.best_score_,
    "test_r2": test_r2,
    "test_mae": test_mae,
    "target": "log1p(positive)",
    "dataset": "FronkonGames Steam Games Dataset"
}

model_path = Path("../models/reviews_model.joblib")

joblib.dump(model_artifact, model_path)

print("Model saved successfully!")
print("Location:", model_path)

Model saved successfully!
Location: ..\models\reviews_model.joblib


In [26]:
# Load the saved model

loaded_artifact = joblib.load(model_path)

loaded_model = loaded_artifact["model"]

print("Loaded model:", loaded_artifact["model_name"])
print("Features:", loaded_artifact["features"])
print("Test MAE:", loaded_artifact["test_mae"])

Loaded model: Random Forest Regressor
Features: ['price', 'windows', 'mac', 'linux', 'required_age', 'categories_count', 'genres_count']
Test MAE: 1.5767058932295228


## M7 - Review Prediction Model Comparison

Three regression models were compared using the same five-fold cross-validation setup: Linear Regression, Decision Tree Regressor and Random Forest Regressor.

RandomizedSearchCV was used to find suitable hyperparameters for the Decision Tree and Random Forest models. The Random Forest achieved the highest mean cross-validation R² score of 0.2866 and was selected as the winning model.

### Cross-validation results

| Model | Mean R² | Standard deviation |
|---|---:|---:|
| Linear Regression | 0.2141 | 0.0066 |
| Decision Tree | 0.2691 | 0.0090 |
| Random Forest | 0.2866 | 0.0090 |

### Final test evaluation

The selected Random Forest was trained on the combined training and validation data and evaluated on the test dataset.

| Metric | M3 Linear Regression | M7 Random Forest |
|---|---:|---:|
| Test R² | 0.2179 | 0.2921 |
| Test MAE | 1.7118 | 1.5767 |

The Random Forest achieved better results than the original Linear Regression model, with a higher R² and lower MAE.

### Selected model configuration

- Model: Random Forest Regressor
- Number of estimators: 100
- Maximum depth: None
- Maximum features: 0.7
- Minimum samples leaf: 10
- Minimum samples split: 12
- Target: `np.log1p(positive)`
- Features: price, windows, mac, linux, required_age, categories_count and genres_count

### Saved model

The winning model was saved using Joblib as `models/reviews_model.joblib`. The saved artifact includes the trained model, feature names, selected hyperparameters and evaluation results.

The test MAE is also recorded in `app/app.py` as `REVIEWS_MAE` for future use in the Streamlit application.